In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.auth import default
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances_argmin_min
from pandas_gbq import to_gbq
from sklearn.cluster import KMeans
from datetime import datetime
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# import the latest data
project_id = 'bigtimestudios'
credentials, project_id = default()
client = bigquery.Client(credentials=credentials,project=project_id)

df = client.query(
    '''
    select
    *
    from bigtimestudios.matt_analytics.cluster_data_baseline
    '''
).to_dataframe()



# id non-economy players
df['total'] = df.drop(columns=['user_id','username','ga_id']).sum(axis=1)

# set label for F2P and split out Economy Participants
df_free = df[df['total'] == 0].drop(columns=['total'])
df_free['cluster'] = -1
df_paid = df[df['total'] > 0].drop(columns=['total'])

# # standardize data
scaler = StandardScaler()
df_paid_scaled = scaler.fit_transform(df_paid.drop(columns=['user_id','username','ga_id']))

# apply initial PCA
pca = PCA(n_components=9,random_state=42)
pca.fit_transform(df_paid_scaled)

# Apply PCA with 6 components
pca = PCA(n_components=6)
df_pca = pca.fit_transform(df_paid_scaled)

#Apply K-Means Clustering with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42)
df_clusters = kmeans.fit_predict(df_pca)

# Add cluster labels back to the DataFrame
df_paid['cluster'] = df_clusters

## get new data
df_new = client.query('''
  select
  *
  from bigtimestudios.matt_analytics.cluster_data_last_30
  ''').to_dataframe()

# split out F2P from paying players
df_new['total'] = df_new.drop(columns=['user_id','username','ga_id']).sum(axis=1)

# set label for F2P and split out Economy Participants
df_new_free = df_new[df_new['total'] == 0].drop(columns=['total'])
df_new_free['cluster'] = -1
df_new_paid = df_new[df_new['total'] > 0].drop(columns=['total'])



def project_and_assign(new_data, pca, kmeans, scaler):
    """
    Scale new data using the pre-fitted scaler, project onto existing PCA components,
    and assign to the nearest centroid.

    Args:
        new_data (DataFrame): New raw data (before PCA).
        pca (PCA model): Trained PCA model with saved components.
        kmeans (KMeans model): Trained KMeans model with saved centroids.
        scaler (StandardScaler): The pre-fitted scaler used on the original data.

    Returns:
        assigned_clusters (ndarray): Cluster labels for the new data.
    """
    # drop non-numeric columns
    cols_to_drop = ['user_id', 'username', 'ga_id','cluster']
    new_data_numeric = new_data.drop(columns=cols_to_drop, errors='ignore')

    # use same scaler to transform new data
    new_data_scaled = scaler.transform(new_data_numeric)  # Do NOT fit again, just transform

    # project scaled data onto the PCA space
    new_data_pca = pca.transform(new_data_scaled)

    # compute distances to centroids
    distances = cdist(new_data_pca[:, :6], kmeans.cluster_centers_, metric='euclidean')

    # assign each point to the nearest centroid
    assigned_clusters = np.argmin(distances, axis=1)

    return assigned_clusters


# apply function
df_new_paid['cluster'] = project_and_assign(df_new_paid, pca, kmeans, scaler)

# function to name
def cluster_name(cluster_number):
  if cluster_number == -1:
    return 'F2P'
  elif cluster_number == 0:
    return 'Mob Farmers' ##Low Econ Participants'
  elif cluster_number == 1:
    return 'Cracked Hourglass Farmers'  ##'Regular Dungeon Farmers'
  elif cluster_number == 2:
    return 'Epoch Chest Farmers' ##'Mega Farmers'
  else:
    return 'Unknown'

# rejoin with F2P users
df_new_total = pd.concat([df_new_free, df_new_paid])
df_new_total['cluster_name'] = df_new_total['cluster'].apply(cluster_name)

# save to bq
project_id = "bigtimestudios"
dataset_id = "PlayerAnalytics"
table_id = "Last30PlayerClusters"

# full BigQuery Table Reference
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# import run date
df_new_total['run_date'] = datetime.now()

# limit columns for upload
user_labels = df_new_total[['run_date','user_id','ga_id','cluster','cluster_name']]
to_gbq(user_labels, table_ref, project_id=project_id, if_exists="append")